<a href="https://colab.research.google.com/github/Saxena-1704/Audio-Deepfake-Detection/blob/main/Audio_Deepfake_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import csv
import time

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.models import vgg19, VGG19_Weights

from tqdm.auto import tqdm


# ============================================================
# 2. CONFIGURATION
# ============================================================

CONFIG = {

    # --------------------------------------------------------
    # Dataset
    # --------------------------------------------------------
    "data_dir": "/content/drive/MyDrive/for-2seconds-mels",

    "train_dir": "training",
    "val_dir": "validation",

    # IMPORTANT:
    # testing directory is intentionally NOT included here.
    # We will use it only for final evaluation later.

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------
    "batch_size": 264,
    "num_epochs": 15,

    "learning_rate": 1e-4,
    "weight_decay": 1e-4,

    # --------------------------------------------------------
    # DataLoader
    # --------------------------------------------------------
    "num_workers": 2,
    "pin_memory": True,

    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------
    "output_dir": "/content/drive/MyDrive/vgg19_real_fake",

    "best_model_name": "best_vgg19_classifier.pth",
    "log_file": "training_log.csv",

    # --------------------------------------------------------
    # Reproducibility
    # --------------------------------------------------------
    "seed": 42,
}


# ============================================================
# 3. REPRODUCIBILITY
# ============================================================

torch.manual_seed(CONFIG["seed"])

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 60)
print("DEVICE")
print("=" * 60)

print(f"Device: {device}")

if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}"
    )

print()


# ============================================================
# 5. OUTPUT DIRECTORY
# ============================================================

output_dir = Path(
    CONFIG["output_dir"]
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

best_model_path = (
    output_dir
    / CONFIG["best_model_name"]
)

log_path = (
    output_dir
    / CONFIG["log_file"]
)


# ============================================================
# 6. VGG19 PREPROCESSING
# ============================================================

weights = VGG19_Weights.DEFAULT

transform = weights.transforms()


# ============================================================
# 7. TRAINING DATASET
# ============================================================

train_path = (
    Path(CONFIG["data_dir"])
    / CONFIG["train_dir"]
)

# Check if training path exists and list its content
print("Checking training data directory...")
if not train_path.exists():
    print(f"Error: Training directory does not exist: {train_path}")
elif not any(train_path.iterdir()):
    print(f"Error: Training directory is empty: {train_path}")
else:
    print(f"Contents of {train_path}:")
    for item in train_path.iterdir():
        print(f"  - {item.name}")
        if item.is_dir():
            sub_items = list(item.iterdir())
            if not sub_items:
                print(f"    - Directory '{item.name}' is empty.")
            else:
                print(f"    - First 5 items in '{item.name}':")
                for i, sub_item in enumerate(sub_items[:5]):
                    print(f"      - {sub_item.name}")
                if len(sub_items) > 5:
                    print(f"      - ... ({len(sub_items) - 5} more items)")

train_dataset = datasets.ImageFolder(
    root=str(train_path),
    transform=transform,
)


# ============================================================
# 8. VALIDATION DATASET
# ============================================================

val_path = (
    Path(CONFIG["data_dir"])
    / CONFIG["val_dir"]
)

# Check if validation path exists and list its content
print("Checking validation data directory...")
if not val_path.exists():
    print(f"Error: Validation directory does not exist: {val_path}")
elif not any(val_path.iterdir()):
    print(f"Error: Validation directory is empty: {val_path}")
else:
    print(f"Contents of {val_path}:")
    for item in val_path.iterdir():
        print(f"  - {item.name}")
        if item.is_dir():
            sub_items = list(item.iterdir())
            if not sub_items:
                print(f"    - Directory '{item.name}' is empty.")
            else:
                print(f"    - First 5 items in '{item.name}':")
                for i, sub_item in enumerate(sub_items[:5]):
                    print(f"      - {sub_item.name}")
                if len(sub_items) > 5:
                    print(f"      - ... ({len(sub_items) - 5} more items)")

val_dataset = datasets.ImageFolder(
    root=str(val_path),
    transform=transform,
)


# ============================================================
# 9. DATASET INFORMATION
# ============================================================

print("=" * 60)
print("DATASET")
print("=" * 60)

print(
    f"Training directory   : {train_path}"
)

print(
    f"Validation directory : {val_path}"
)

print()

print(
    f"Training images      : "
    f"{len(train_dataset):,}"
)

print(
    f"Validation images    : "
    f"{len(val_dataset):,}"
)

print()

print(
    f"Classes              : "
    f"{train_dataset.classes}"
)

print(
    f"Class mapping        : "
    f"{train_dataset.class_to_idx}"
)

print()


# ============================================================
# 10. DATALOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,

    batch_size=CONFIG["batch_size"],

    shuffle=True,

    num_workers=CONFIG["num_workers"],

    pin_memory=(
        CONFIG["pin_memory"]
        and torch.cuda.is_available()
    ),
)


val_loader = DataLoader(
    val_dataset,

    batch_size=CONFIG["batch_size"],

    shuffle=False,

    num_workers=CONFIG["num_workers"],

    pin_memory=(
        CONFIG["pin_memory"]
        and torch.cuda.is_available()
    ),
)


# ============================================================
# 11. LOAD PRETRAINED VGG19
# ============================================================

print("=" * 60)
print("LOADING VGG19")
print("=" * 60)

model = vgg19(
    weights=weights
)


# ============================================================
# 12. FREEZE EVERYTHING
# ============================================================

for param in model.parameters():

    param.requires_grad = False


# ============================================================
# 13. REPLACE FINAL CLASSIFIER
# ============================================================

# Original:
#
# Linear(4096, 1000)
#
# New:
#
# Linear(4096, 2)

model.classifier[6] = nn.Linear(
    model.classifier[6].in_features,
    2
)


# The newly created layer is trainable by default.
#
# Therefore:
#
# classifier[0] -> frozen
# classifier[3] -> frozen
# classifier[6] -> trainable
#
# features -> completely frozen


# ============================================================
# 14. MOVE MODEL TO DEVICE
# ============================================================

model = model.to(device)


# ============================================================
# 15. VERIFY TRAINABLE PARAMETERS
# ============================================================

print()
print("=" * 60)
print("TRAINABLE PARAMETERS")
print("=" * 60)

total_params = 0
trainable_params = 0

for name, param in model.named_parameters():

    total_params += param.numel()

    if param.requires_grad:

        trainable_params += param.numel()

        print(name)


print()

print(
    f"Total parameters     : "
    f"{total_params:,}"
)

print(
    f"Trainable parameters : "
    f"{trainable_params:,}"
)

print()


# ============================================================
# 16. LOSS FUNCTION
# ============================================================

criterion = nn.CrossEntropyLoss()


# ============================================================
# 17. OPTIMIZER
# ============================================================

# IMPORTANT:
# Only the trainable final classifier is passed
# to the optimizer.

optimizer = optim.Adam(
    model.classifier[6].parameters(),

    lr=CONFIG["learning_rate"],

    weight_decay=CONFIG["weight_decay"],
)


# ============================================================
# 18. CREATE CSV LOG
# ============================================================

with open(
    log_path,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "epoch",
        "train_loss",
        "train_accuracy",
        "val_loss",
        "val_accuracy",
        "epoch_time_seconds",
    ])


# ============================================================
# 19. TRAIN ONE EPOCH
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False,
    )

    for images, labels in progress_bar:

        # ----------------------------------------------------
        # Move data to GPU / CPU
        # ----------------------------------------------------

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        # ----------------------------------------------------
        # Clear gradients
        # ----------------------------------------------------

        optimizer.zero_grad()

        # ----------------------------------------------------
        # Forward pass
        # ----------------------------------------------------

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        # ----------------------------------------------------
        # Backward pass
        # ----------------------------------------------------

        loss.backward()

        # ----------------------------------------------------
        # Update ONLY final classifier
        # ----------------------------------------------------

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        batch_size = labels.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        _, predicted = torch.max(
            outputs,
            dim=1
        )

        total += batch_size

        correct += (
            predicted == labels
        ).sum().item()

        # ----------------------------------------------------
        # Progress bar
        # ----------------------------------------------------

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# 20. VALIDATION
# ============================================================

def validate(
    model,
    loader,
    criterion,
    device,
):

    # Evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # IMPORTANT:
    # No gradients are calculated during validation.

    with torch.no_grad():

        progress_bar = tqdm(
            loader,
            desc="Validation",
            leave=False,
        )

        for images, labels in progress_bar:

            # ------------------------------------------------
            # Move data to device
            # ------------------------------------------------

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            # ------------------------------------------------
            # Forward pass only
            # ------------------------------------------------

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            # ------------------------------------------------
            # Statistics
            # ------------------------------------------------

            batch_size = labels.size(0)

            running_loss += (
                loss.item() * batch_size
            )

            _, predicted = torch.max(
                outputs,
                dim=1
            )

            total += batch_size

            correct += (
                predicted == labels
            ).sum().item()

    epoch_loss = (
        running_loss / total
    )

    epoch_accuracy = (
        correct / total
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# ============================================================
# 21. TRAINING LOOP
# ============================================================

print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

print()
print(
    "Training data  : training/"
)

print(
    "Validation data: validation/"
)

print(
    "Testing data   : NOT USED"
)

print()


# Best validation loss
best_val_loss = float("inf")


for epoch in range(
    CONFIG["num_epochs"]
):

    epoch_start = time.time()

    print()
    print("=" * 60)

    print(
        f"Epoch "
        f"{epoch + 1}/"
        f"{CONFIG['num_epochs']}"
    )

    print("=" * 60)

    # ========================================================
    # TRAIN
    # ========================================================

    train_loss, train_accuracy = train_one_epoch(
        model=model,

        loader=train_loader,

        criterion=criterion,

        optimizer=optimizer,

        device=device,
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    val_loss, val_accuracy = validate(
        model=model,

        loader=val_loader,

        criterion=criterion,

        device=device,
    )


    # ========================================================
    # EPOCH TIME
    # ========================================================

    epoch_time = (
        time.time()
        - epoch_start
    )


    # ========================================================
    # LOG RESULTS
    # ========================================================

    print()

    print(
        f"Train Loss      : "
        f"{train_loss:.4f}"
    )

    print(
        f"Train Accuracy  : "
        f"{train_accuracy * 100:.2f}%"
    )

    print(
        f"Validation Loss : "
        f"{val_loss:.4f}"
    )

    print(
        f"Validation Acc  : "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Epoch Time      : "
        f"{epoch_time:.2f} seconds"
    )


    # ========================================================
    # SAVE LOG
    # ========================================================

    with open(
        log_path,
        "a",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            epoch + 1,
            train_loss,
            train_accuracy,
            val_loss,
            val_accuracy,
            epoch_time,
        ])


    # ========================================================
    # SAVE BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            {
                "epoch":
                    epoch + 1,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "val_loss":
                    val_loss,

                "val_accuracy":
                    val_accuracy,

                "class_to_idx":
                    train_dataset.class_to_idx,
            },

            best_model_path,
        )

        print()
        print(
            "✓ New best model saved!"
        )


# ============================================================
# 22. TRAINING COMPLETE
# ============================================================

print()
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    f"Best validation loss: "
    f"{best_val_loss:.4f}"
)

print()

print(
    f"Best model:"
    f"\n{best_model_path}"
)

print()

print(
    f"Training log:"
    f"\n{log_path}"
)